# FPL Team Monitor

Weekly health-check for your FPL squad. Each run:

1. Fetches the latest player data from the FPL API
2. Saves a snapshot to a local SQLite file (`data/fpl_snapshots.sqlite` - no database server needed)
3. Diffs against the previous snapshot to compute **week-over-week % change in transfer activity** (`transfers_in_event` / `transfers_out_event`) for your squad and watchlist
4. Flags injuries/suspensions, price changes, and likely price moves
5. Suggests a captain for the upcoming gameweek from fixture difficulty + form

**How the week-over-week % works:** the FPL API's transfer counters accumulate within the current gameweek and reset at each deadline. Snapshot once a week - ideally the same day each week - and the % change compares this week's transfer flow against last week's, which is the momentum signal that anticipates price moves and bandwagons. The first run just saves a baseline; percentages appear from the second run.

## Install Required Packages

In [ ]:
# Install required packages
import sys
!{sys.executable} -m pip install requests pandas

## Imports

In [ ]:
import pandas as pd
from datetime import date

# Shared modules (same folder as this notebook)
import fpl_data as fpl_api
import fpl_monitor

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Configuration

List your 15-man squad and any transfer targets you're watching. Names can be either the short name shown on fantasy.premierleague.com (e.g. `M.Salah`) or the full name (e.g. `Mohamed Salah`) - unmatched or ambiguous names are reported when they're resolved below.

In [ ]:
# Your current 15-man squad (from the team selector notebook or your live FPL team)
MY_SQUAD = [
    # "M.Salah",
    # "Erling Haaland",
]

# Transfer targets to track even when they aren't big market movers
WATCHLIST = [
    # "Player Name",
]

# Where weekly snapshots are stored
SNAPSHOT_DB = 'data/fpl_snapshots.sqlite'

# Report thresholds
SWING_THRESHOLD_PCT = 50.0     # WoW % change in transfer flow considered "large"
MIN_TRANSFER_VOLUME = 5000     # ignore movers below this many transfers this GW (noise)
PRICE_RISE_NET = 40000         # net transfers-in above this -> possible price rise
PRICE_FALL_NET = -40000        # net transfers-in below this -> possible price fall

## Fetch, Diff and Snapshot

Fetches current data, diffs it against the previous snapshot, then saves today's snapshot. Re-running on the same day replaces today's snapshot, so you always diff against the previous week - never against a run from earlier today.

In [ ]:
bootstrap = fpl_api.fetch_bootstrap_static()
current_df = fpl_monitor.bootstrap_to_snapshot_df(bootstrap)

upcoming = fpl_monitor.get_upcoming_event(bootstrap)
event_id = upcoming['id'] if upcoming else None
if upcoming:
    print(f"Upcoming gameweek: {upcoming['name']} (deadline {upcoming['deadline_time']})")

previous_df, previous_date = fpl_monitor.load_previous_snapshot(
    SNAPSHOT_DB, exclude_date=date.today().isoformat())
changes_df = fpl_monitor.compute_changes(current_df, previous_df)

snapshot_date = fpl_monitor.save_snapshot(current_df, SNAPSHOT_DB, event_id=event_id)
if previous_date:
    print(f"Snapshot saved for {snapshot_date}; comparing against {previous_date}.")
else:
    print(f"First snapshot saved for {snapshot_date} - week-over-week figures "
          "will appear from next week's run.")

# Resolve configured names to player ids (typos/ambiguities reported here)
squad_ids = fpl_monitor.resolve_player_ids(current_df, MY_SQUAD)
watchlist_ids = fpl_monitor.resolve_player_ids(current_df, WATCHLIST)
if not squad_ids:
    print("\nTip: fill in MY_SQUAD above to get squad alerts, momentum and a captain pick.")

## Weekly Report

In [ ]:
fpl_monitor.build_report(
    changes_df, squad_ids, watchlist_ids,
    swing_threshold_pct=SWING_THRESHOLD_PCT,
    min_transfer_volume=MIN_TRANSFER_VOLUME,
    price_rise_net=PRICE_RISE_NET,
    price_fall_net=PRICE_FALL_NET,
)

## Captain Suggestion

Ranks your squad for the upcoming gameweek by `base score x fixture ease`. Base score is `form` once the season is underway; before any matches it falls back to points-per-game, then ownership %. Fixture ease comes from FPL's difficulty ratings (1 easy - 5 hard); double gameweeks count both fixtures, blank gameweeks score 0, and flagged (injured/suspended/doubtful) players are zeroed.

In [ ]:
if squad_ids and event_id:
    fixtures = fpl_api.fetch_fixtures(event_id=event_id)
    team_names = {t['id']: t['name'] for t in bootstrap['teams']}
    captain_df = fpl_monitor.suggest_captain(
        current_df, squad_ids, fixtures, event_id, team_names=team_names)

    picks = captain_df[captain_df['captain_score'] > 0]
    if len(picks) >= 2:
        print(f"Suggested captain:      {picks.iloc[0]['web_name']} ({picks.iloc[0]['fixture']})")
        print(f"Suggested vice-captain: {picks.iloc[1]['web_name']} ({picks.iloc[1]['fixture']})\n")
    elif picks.empty:
        print("No available player has a fixture this gameweek - check the table below.\n")

    print(captain_df.to_string(index=False))
else:
    print("Fill in MY_SQUAD (and make sure a gameweek is upcoming) to get a captain suggestion.")

## Notes

- Run this once a week, on a consistent day (e.g. every Friday, or right after each deadline) so the week-over-week percentages compare like with like.
- Price-move flags are a heuristic based on net transfer volume - FPL's real price algorithm is unpublished, so treat them as "check tonight", not certainties.
- Snapshots accumulate in `data/fpl_snapshots.sqlite`; the file is small and safe to delete if you ever want a fresh start.
- To turn a flagged player into a transfer decision, re-run `FPL_Team_Selector_Consolidated.ipynb` with your current squad constraints and compare.